In [1]:
from pyspark.sql import SparkSession
from pyspark import SparkConf
from pyspark.sql import functions as F, types as T
from pyspark.sql.window import Window
from pathlib import Path

SEED = 42
DECAY_GAMMA = 0.03
EB_KAPPA = 500

# TODO: 
TX_PATH = "./data/curated/merchant_transactions"
PROB_PATH = "./data/tables/merchant_data/consumer_fraud_probability.csv"
OUTPUT_DIR = "./artifacts/fraud_outputs"

conf = (
    SparkConf()
    .setAppName("fraud_prob_pipeline")
    .set("spark.sql.adaptive.enabled", "true")
    .set("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .set("spark.sql.shuffle.partitions", "200")
    .set("spark.driver.memory", "6g")
    .set("spark.executor.memory", "4g")
)

spark = (
    SparkSession.builder.master("local[*]").config(conf=conf).getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)
print("TX_PATH:", TX_PATH)
print("PROB_PATH:", PROB_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/06 01:12:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 4.0.0
TX_PATH: ./data/curated/merchant_transactions
PROB_PATH: ./data/tables/merchant_data/consumer_fraud_probability.csv
OUTPUT_DIR: ./artifacts/fraud_outputs


### Baseline Approach (fast path)


In [2]:
# Tunables
SEED = 42
A_USER = 100.0
B_MERCH = 200.0
KAPPA = 500.0
LAMBDA = 1.0


In [3]:
# Quick loader: define transactions and probs before direct join
schema_tx = T.StructType([
    T.StructField("merchant_abn", T.LongType(), True),
    T.StructField("user_id", T.LongType(), True),
    T.StructField("dollar_value", T.DoubleType(), True),
    T.StructField("order_datetime", T.DateType(), True),
    T.StructField("business", T.StringType(), True),
    T.StructField("biz_tags", T.StringType(), True),
    T.StructField("rev_band", T.StringType(), True),
    T.StructField("take_rate", T.StringType(), True),
])
schema_prob = T.StructType([
    T.StructField("user_id", T.LongType(), True),
    T.StructField("order_datetime", T.DateType(), True),
    T.StructField("fraud_probability", T.DoubleType(), True),
])

transactions = spark.read.schema(schema_tx).parquet(TX_PATH)
probs = spark.read.option("header", True).schema(schema_prob).csv(PROB_PATH)



In [4]:
# Direct join to get p_direct (align to DATE)
probs_d = probs.select(
    "user_id",
    F.to_date("order_datetime").alias("order_datetime"),
    F.col("fraud_probability").alias("p_direct")
)

matched = transactions.join(F.broadcast(
    probs_d), ["user_id", "order_datetime"], "left")
print("Direct matches:", matched.filter(F.col("p_direct").isNotNull()).count())

Direct matches: 54764


In [5]:
# Tiny baselines from matched rows

matched_only = matched.filter(F.col("p_direct").isNotNull()).select("user_id", "merchant_abn", "p_direct")

user_base = matched_only.groupBy("user_id").agg(
    F.count(F.lit(1)).alias("n_user"),
    F.avg("p_direct").alias("p_user")
)

merch_base = matched_only.groupBy("merchant_abn").agg(
    F.count(F.lit(1)).alias("n_merch"),
    F.avg("p_direct").alias("p_merch")
)

mu_row = matched_only.agg(F.avg("p_direct").alias("mu")).first()
MU = float(mu_row["mu"]) if mu_row and mu_row["mu"] is not None else 0.01
print("Baselines ready. MU=", MU)


Baselines ready. MU= 14.96808671867667


### Load & Basic Hygiene


In [6]:
# Read inputs with enforced schemas

schema_tx = T.StructType([
    T.StructField("merchant_abn", T.LongType(), True),
    T.StructField("user_id", T.LongType(), True),
    T.StructField("dollar_value", T.DoubleType(), True),
    T.StructField("order_datetime", T.DateType(), True),
    T.StructField("business", T.StringType(), True),
    T.StructField("biz_tags", T.StringType(), True),
    T.StructField("rev_band", T.StringType(), True),
    T.StructField("take_rate", T.StringType(), True),
])

schema_prob = T.StructType([
    T.StructField("user_id", T.LongType(), True),
    T.StructField("order_datetime", T.DateType(), True),
    T.StructField("fraud_probability", T.DoubleType(), True),
])

# Load
transactions = spark.read.schema(schema_tx).parquet(TX_PATH)
probs = spark.read.option("header", True).schema(schema_prob).csv(PROB_PATH)
# Normalize to [0,1] if file stores percents (e.g., 97.6 -> 0.976)
probs = probs.withColumn(
    "fraud_probability",
    F.when(F.col("fraud_probability") > 1.0, F.col("fraud_probability") / F.lit(100.0)).otherwise(F.col("fraud_probability"))
)

# Trim/clean
transactions = (
    transactions
    .withColumn("biz_tags", F.trim(F.regexp_replace(F.col("biz_tags"), "\s+", " ")))
    .withColumn("rev_band", F.trim(F.col("rev_band")))
    .withColumn("take_rate", F.trim(F.col("take_rate")))
)

# Parse take_rate -> numeric percent when string like "2.5%" else try cast
transactions = transactions.withColumn(
    "take_rate_num",
    F.when(F.col("take_rate").rlike(r"^[0-9.]+%$"), F.regexp_replace("take_rate", "%", "").cast("double") / 100.0)
     .when(F.col("take_rate").rlike(r"^[0-9.]+$"), F.col("take_rate").cast("double"))
     .otherwise(F.lit(None).cast("double"))
)

# Drop duplicates
transactions = transactions.dropDuplicates(["merchant_abn", "user_id", "order_datetime", "dollar_value"]).cache()
probs = probs.dropDuplicates(["user_id", "order_datetime"]).cache()

print("Transactions:", transactions.count())
print("Probs rows:", probs.count())


Transactions: 13293831
Probs rows: 34765


In [7]:
transactions.show(20)
probs.show(20)

+------------+-------+------------------+--------------+--------------------+--------------------+--------+---------+-------------+
|merchant_abn|user_id|      dollar_value|order_datetime|            business|            biz_tags|rev_band|take_rate|take_rate_num|
+------------+-------+------------------+--------------+--------------------+--------------------+--------+---------+-------------+
| 10023283211|   4867|328.86441335410353|    2022-01-14|       Felis Limited|furniture, home f...|       e|     0.18|         0.18|
| 10023283211|   5932|183.11850116341057|    2022-07-07|       Felis Limited|furniture, home f...|       e|     0.18|         0.18|
| 10023283211|   7304|36.605324819002185|    2022-07-18|       Felis Limited|furniture, home f...|       e|     0.18|         0.18|
| 10023283211|  10618| 432.6380561623335|    2022-01-17|       Felis Limited|furniture, home f...|       e|     0.18|         0.18|
| 10023283211|  14038| 19.71928451317702|    2022-09-18|       Felis Limited

### Tier A: Direct probability matches


In [8]:
# Left join on (user_id, order_datetime) to get p_direct

transactions = transactions.withColumn("order_date", F.col("order_datetime"))
probs = probs.withColumn("order_date", F.col("order_datetime")).drop("order_datetime")

joined_a = transactions.join(
    probs.select("user_id", "order_date", F.col("fraud_probability").alias("p_direct")),
    ["user_id", "order_date"],
    "left",
)

print("Tier A: p_direct non-null:", joined_a.filter(F.col("p_direct").isNotNull()).count())
joined_a.cache()


Tier A: p_direct non-null: 54636


DataFrame[user_id: bigint, order_date: date, merchant_abn: bigint, dollar_value: double, order_datetime: date, business: string, biz_tags: string, rev_band: string, take_rate: string, take_rate_num: double, p_direct: double]

### Tier B: User-level propensity with time decay


In [9]:
# Compute p_user_decay for rows without p_direct

# Prepare per-user probability history as (user_id, d_k, p_k)
prob_hist = probs.select(
    "user_id", F.col("order_date").alias("d_k"), F.col("fraud_probability").alias("p_k")
)

# For efficiency: join only for users present in transactions lacking p_direct
users_needing = joined_a.filter(F.col("p_direct").isNull()).select("user_id").distinct()

cand = (
    joined_a.select("user_id", "order_date")
    .join(users_needing, "user_id", "inner")
    .join(prob_hist, "user_id", "inner")
)

# Compute weights w_k = exp(-gamma * |t - d_k|) in days
diff_days = F.abs(F.datediff(F.col("order_date"), F.col("d_k")))
cand = cand.withColumn("w_k", F.exp(-DECAY_GAMMA * diff_days))

# Aggregate per (user_id, order_date)
p_user_decay_df = (
    cand.groupBy("user_id", "order_date")
    .agg(
        (F.sum(F.col("p_k") * F.col("w_k")) / F.sum(F.col("w_k"))).alias("p_user_decay")
    )
)

# Join back onto joined_a, only where p_direct is null
joined_b = (
    joined_a
    .join(p_user_decay_df, ["user_id", "order_date"], "left")
    .withColumn("p_user_decay", F.when(F.col("p_direct").isNull(), F.col("p_user_decay")).otherwise(F.lit(None)))
)

print("Tier B: filled via decay:", joined_b.filter(F.col("p_user_decay").isNotNull()).count())
joined_b.cache()


Tier B: filled via decay: 11056363


DataFrame[user_id: bigint, order_date: date, merchant_abn: bigint, dollar_value: double, order_datetime: date, business: string, biz_tags: string, rev_band: string, take_rate: string, take_rate_num: double, p_direct: double, p_user_decay: double]

In [10]:
joined_b.show(20)

+-------+----------+------------+------------------+--------------+--------------------+--------------------+--------+---------+-------------+--------+-------------------+
|user_id|order_date|merchant_abn|      dollar_value|order_datetime|            business|            biz_tags|rev_band|take_rate|take_rate_num|p_direct|       p_user_decay|
+-------+----------+------------+------------------+--------------+--------------------+--------------------+--------+---------+-------------+--------+-------------------+
|      1|2021-03-21| 72472909171| 26.84427554025195|    2021-03-21|   Nullam Consulting|digital goods: bo...|       a|     6.33|         6.33|    NULL|0.09805431136520959|
|      1|2021-03-21| 86010199872|218.49116722954264|    2021-03-21|  Orci In Foundation|computer programm...|       c|     2.40|          2.4|    NULL|0.09805431136520959|
|      1|2021-09-07| 82065156333| 7.894702563047845|    2021-09-07|Nascetur Ridiculu...|tent and awning s...|       a|     5.88|         5.8

### Tier C: Model to impute remaining probabilities (soft-label regression)


### Tier C Features


In [11]:
# Feature engineering for modeling

# Base for modeling
base = joined_b.withColumn(
    "p_label", F.coalesce(F.col("p_direct"), F.lit(None))  # we only train where p_direct present
)

# Transaction-level
base = base.withColumn("log_amount", F.log1p(F.col("dollar_value"))) \
           .withColumn("dow", F.dayofweek("order_date")) \
           .withColumn("month", F.month("order_date"))

# User windows (90d)
w_user_90 = (
    Window.partitionBy("user_id").orderBy(F.col("order_date").cast("timestamp").cast("long")).rangeBetween(-90*86400, 0)
)
base = base.withColumn("user_txn_count_90d", F.count(F.lit(1)).over(w_user_90)) \
           .withColumn("user_sum_90d", F.sum("dollar_value").over(w_user_90)) \
           .withColumn("user_avg_amount_90d", F.avg("dollar_value").over(w_user_90))

# Days since previous txn per user
w_user_prev = Window.partitionBy("user_id").orderBy("order_date")
base = base.withColumn("prev_date", F.lag("order_date").over(w_user_prev)) \
           .withColumn("user_days_since_prev", F.datediff("order_date", "prev_date")) \
           .drop("prev_date")

# Merchant windows (90d)
w_merch_90 = (
    Window.partitionBy("merchant_abn").orderBy(F.col("order_date").cast("timestamp").cast("long")).rangeBetween(-90*86400, 0)
)
base = base.withColumn("m_txn_count_90d", F.count(F.lit(1)).over(w_merch_90)) \
           .withColumn("m_sum_90d", F.sum("dollar_value").over(w_merch_90)) \
           .withColumn("m_avg_amount_90d", F.avg("dollar_value").over(w_merch_90))

# Impute missing numeric features to avoid NaN/Inf in vectors
numeric_fill = [
    "log_amount", "user_txn_count_90d", "user_sum_90d", "user_avg_amount_90d",
    "user_days_since_prev", "m_txn_count_90d", "m_sum_90d", "m_avg_amount_90d", "take_rate_num"
]
base = base.fillna(0, subset=numeric_fill)

# Categorical encodings: biz_tags, rev_band
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline

indexers = [
    StringIndexer(inputCol="rev_band", outputCol="rev_band_idx", handleInvalid="keep"),
    StringIndexer(inputCol="biz_tags", outputCol="biz_tags_idx", handleInvalid="keep"),
]
encoders = [
    OneHotEncoder(inputCols=["rev_band_idx", "biz_tags_idx"], outputCols=["rev_band_oh", "biz_tags_oh"])
]

feats = [
    "log_amount", "dow", "month", "user_txn_count_90d", "user_sum_90d", "user_avg_amount_90d",
    "user_days_since_prev", "m_txn_count_90d", "m_sum_90d", "m_avg_amount_90d", "take_rate_num"
]

assembler = VectorAssembler(inputCols=feats + ["rev_band_oh", "biz_tags_oh"], outputCol="features", handleInvalid="keep")

prep_pipeline = Pipeline(stages=indexers + encoders + [assembler])

# Labeled training data: where p_direct is available
labeled = base.filter(F.col("p_direct").isNotNull()).cache()
print("Labeled rows:", labeled.count())


Labeled rows: 54636


In [12]:
# Train LR and GBT; evaluate and calibrate

from pyspark.ml.regression import LinearRegression, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline

prepared = prep_pipeline.fit(labeled).transform(labeled)

train_df, valid_df = prepared.randomSplit([0.8, 0.2], seed=SEED)

lr = LinearRegression(labelCol="p_direct", featuresCol="features", elasticNetParam=0.5, regParam=0.1)
gbt = GBTRegressor(labelCol="p_direct", featuresCol="features", maxDepth=6, maxIter=60, stepSize=0.1, subsamplingRate=0.8)

lr_model = lr.fit(train_df)
gbt_model = gbt.fit(train_df)

pred_lr = lr_model.transform(valid_df)
pred_gbt = gbt_model.transform(valid_df)

# Evaluate MAE and Brier (MSE on [0,1])
mae_eval = RegressionEvaluator(labelCol="p_direct", predictionCol="prediction", metricName="mae")
mse_eval = RegressionEvaluator(labelCol="p_direct", predictionCol="prediction", metricName="mse")

def _clip01(col):
    return F.when(F.col(col) < 0, 0.0).when(F.col(col) > 1, 1.0).otherwise(F.col(col))

def evaluate(df, label="p_direct", pred="prediction"):
    tmp = df.withColumn("lbl", _clip01(label)).withColumn("prd", _clip01(pred))
    mae   = RegressionEvaluator(labelCol="lbl", predictionCol="prd", metricName="mae").evaluate(tmp)
    brier = RegressionEvaluator(labelCol="lbl", predictionCol="prd", metricName="mse").evaluate(tmp)
    return mae, brier

metrics = {
    "lr_mae": evaluate(pred_lr)[0],
    "lr_brier": evaluate(pred_lr)[1],
    "gbt_mae": evaluate(pred_gbt)[0],
    "gbt_brier": evaluate(pred_gbt)[1],
}
print(metrics)

# Choose best model (skip isotonic calibration for speed)
best_is_gbt = metrics["gbt_mae"] <= metrics["lr_mae"]
best_model = gbt_model if best_is_gbt else lr_model


25/10/06 01:16:28 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
25/10/06 01:16:29 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


{'lr_mae': 0.05608904317340927, 'lr_brier': 0.007550851964725374, 'gbt_mae': 0.04445827393810033, 'gbt_brier': 0.005440373250684371}


In [13]:
# Apply best model to unlabeled rows and calibrate

prep_model = prep_pipeline.fit(labeled)
prepared = prep_model.transform(labeled)

# Re-train both on prepared
train_df, valid_df = prepared.randomSplit([0.8, 0.2], seed=SEED)

lr = LinearRegression(labelCol="p_direct", featuresCol="features", elasticNetParam=0.5, regParam=0.1)
gbt = GBTRegressor(labelCol="p_direct", featuresCol="features", maxDepth=6, maxIter=60, stepSize=0.1, subsamplingRate=0.8)

lr_model = lr.fit(train_df)
gbt_model = gbt.fit(train_df)

pred_lr = lr_model.transform(valid_df)
pred_gbt = gbt_model.transform(valid_df)

mae_eval = RegressionEvaluator(labelCol="p_direct", predictionCol="prediction", metricName="mae")
mse_eval = RegressionEvaluator(labelCol="p_direct", predictionCol="prediction", metricName="mse")

metrics = {
    "lr_mae": mae_eval.evaluate(pred_lr),
    "lr_brier": mse_eval.evaluate(pred_lr),
    "gbt_mae": mae_eval.evaluate(pred_gbt),
    "gbt_brier": mse_eval.evaluate(pred_gbt),
}
print("Validation metrics:", metrics)

best_is_gbt = metrics["gbt_mae"] <= metrics["lr_mae"]
best_model = gbt_model if best_is_gbt else lr_model
best_valid = pred_gbt if best_is_gbt else pred_lr

unlabeled = base.filter(F.col("p_direct").isNull() & F.col("p_user_decay").isNull())
prepared_unlabeled = prep_model.transform(unlabeled)

scored_unlabeled = best_model.transform(prepared_unlabeled)

# Clip predictions to [0,1] as calibrated scores
scored_unlabeled = scored_unlabeled.withColumn(
    "p_model_calibrated",
    F.when(F.col("prediction") < 0, 0.0).when(F.col("prediction") > 1, 1.0).otherwise(F.col("prediction"))
)

print("Scored unlabeled rows:", scored_unlabeled.count())


Validation metrics: {'lr_mae': 0.05608904317340927, 'lr_brier': 0.007550851964725374, 'gbt_mae': 0.04445827393810033, 'gbt_brier': 0.005440373250684372}
Scored unlabeled rows: 2182832


In [14]:
# Coalesce probabilities and persist per-transaction output

# Bring together tiers
coalesced = (
    joined_b
    .join(scored_unlabeled.select("merchant_abn", "user_id", "order_date", "p_model_calibrated"), ["merchant_abn", "user_id", "order_date"], "left")
    .withColumn("p_hat", F.coalesce(F.col("p_direct"), F.col("p_user_decay"), F.col("p_model_calibrated")))
)

# Clip to [0,1], fallback to global mean if any remain null
mu = coalesced.select(F.mean("p_hat")).first()[0]
coalesced = coalesced.withColumn("p_hat", F.when(F.col("p_hat").isNull(), F.lit(mu)).otherwise(F.col("p_hat")))
coalesced = coalesced.withColumn("p_hat", F.when(F.col("p_hat") < 0, 0.0).when(F.col("p_hat") > 1, 1.0).otherwise(F.col("p_hat")))

per_tx_out = coalesced.select(
    "merchant_abn", "user_id", F.col("order_date").alias("order_datetime"), "dollar_value", "biz_tags", "rev_band", "take_rate", "p_hat"
)

print("Per-transaction output rows:", per_tx_out.count())


Per-transaction output rows: 13303861


### Merchant-level aggregation & Empirical-Bayes shrinkage


In [15]:
# Aggregate to merchant metrics and EB shrinkage

agg = (
    per_tx_out.groupBy("merchant_abn")
    .agg(
        F.count(F.lit(1)).alias("n_txn"),
        F.sum("dollar_value").alias("sum_amount"),
        F.avg("p_hat").alias("mean_p"),
        F.sum(F.col("p_hat") * F.col("dollar_value")).alias("EFL"),
    )
    .withColumn("EFLR", F.col("EFL") / F.when(F.col("sum_amount") == 0, F.lit(1.0)).otherwise(F.col("sum_amount")))
)

mu = per_tx_out.select(F.avg("p_hat").alias("mu")).first()[0]

agg = agg.withColumn(
    "eb_p",
    (F.col("n_txn") * F.col("mean_p") + F.lit(EB_KAPPA) * F.lit(mu)) / (F.col("n_txn") + F.lit(EB_KAPPA))
)

agg = agg.withColumn(
    "se_mean_p",
    F.sqrt(F.col("mean_p") * (1 - F.col("mean_p")) / F.greatest(F.col("n_txn"), F.lit(1)))
)


In [ ]:
# Top-20 merchants by eb_p with names 
MERCHANTS_PATH = "./data/tables/merchant_data/tbl_merchants.parquet"

top20 = agg.orderBy(F.col("eb_p").desc()).limit(20)

merchants_sel = (
    spark.read.parquet(MERCHANTS_PATH)
    .select("merchant_abn", F.col("name").alias("merchant_name"))
)

(top20
 .join(F.broadcast(merchants_sel), "merchant_abn", "left")
 .select("merchant_abn", "merchant_name", "eb_p")
 .orderBy(F.col("eb_p").desc())
 .show(20, truncate=False))

+------------+-------------------------------+-------------------+
|merchant_abn|merchant_name                  |eb_p               |
+------------+-------------------------------+-------------------+
|19492220327 |Commodo Ipsum Industries       |0.19062574634249185|
|90918180829 |Pharetra Quisque Company       |0.19022919073827627|
|15043504837 |Odio Incorporated              |0.18459481843831577|
|14530561097 |Duis At Inc.                   |0.17681900463482017|
|44345785419 |Phasellus Nulla LLC            |0.17202392057633537|
|66228393506 |Sed Hendrerit Foundation       |0.16920355215018545|
|23709946765 |Faucibus Leo Corp.             |0.16888013028056653|
|86889657711 |Accumsan Corporation           |0.16856452351620232|
|58495294020 |Diam Nunc Associates           |0.16829770044795692|
|73489866331 |Eu Dui Cum Company             |0.16815110743364733|
|57860746842 |Dui Augue PC                   |0.166630390981463  |
|10596295795 |Egestas A Associates           |0.16645534745259

In [ ]:
# Top-20 merchants by eb_p with names

last20 = agg.orderBy(F.col("eb_p").asc()).limit(20)

merchants_sel = (
    spark.read.parquet(MERCHANTS_PATH)
    .select("merchant_abn", F.col("name").alias("merchant_name"))
)

(top20
 .join(F.broadcast(merchants_sel), "merchant_abn", "left")
 .select("merchant_abn", "merchant_name", "eb_p")
 .orderBy(F.col("eb_p").asc())
 .show(20, truncate=False))

+------------+-------------------------------+-------------------+
|merchant_abn|merchant_name                  |eb_p               |
+------------+-------------------------------+-------------------+
|59039508122 |Augue Ut Lacus Foundation      |0.161932445951984  |
|95402237897 |Turpis Company                 |0.16240255892562916|
|29253286472 |Fusce Incorporated             |0.16291366561940082|
|73499119023 |Nisi Dictum Company            |0.164570122742482  |
|99022662131 |Auctor Vitae Aliquet Associates|0.16546068724148946|
|28449116125 |Est Industries                 |0.1655508514402402 |
|43083074133 |Faucibus Leo In LLC            |0.1656191291576802 |
|17523010120 |Urna Convallis Foundation      |0.16578971660545316|
|10596295795 |Egestas A Associates           |0.16645534745259646|
|57860746842 |Dui Augue PC                   |0.166630390981463  |
|73489866331 |Eu Dui Cum Company             |0.16815110743364733|
|58495294020 |Diam Nunc Associates           |0.16829770044795

In [18]:
print("report merchants:", agg.count())

report merchants: 3925


In [23]:
# Read merchant names once and join to agg so merchant_name is available for all rankings
MERCHANTS_PATH = "./data/tables/merchant_data/tbl_merchants.parquet"
merchants_sel = (
    spark.read.parquet(MERCHANTS_PATH)
    .select("merchant_abn", F.col("name").alias("merchant_name"))
)

# Join names to agg -> agg_named
agg_named = agg.join(F.broadcast(merchants_sel), "merchant_abn", "left")

# Top 20 by EB probability
top20 = (
    agg_named
    .orderBy(F.col("eb_p").desc())
    .select("merchant_abn", "merchant_name", "n_txn", "sum_amount", "eb_p", "EFL", "EFLR")
    .limit(20)
)
print("Top 20 merchants (highest eb_p):")
top20.show(20, truncate=False)

# Bottom 20 by EB probability
bottom20 = (
    agg_named
    .orderBy(F.col("eb_p").asc())
    .select("merchant_abn", "merchant_name", "n_txn", "sum_amount", "eb_p", "EFL", "EFLR")
    .limit(20)
)
print("Bottom 20 merchants (lowest eb_p):")
bottom20.show(20, truncate=False)

print("report merchants:", agg_named.count())

MERCHANTS_PATH = "./data/tables/merchant_data/tbl_merchants.parquet"
merchants_sel = (
    spark.read.parquet(MERCHANTS_PATH)
    .select("merchant_abn", F.col("name").alias("merchant_name"))
)

# Join names to agg -> agg_named
agg_named = agg.join(F.broadcast(merchants_sel), "merchant_abn", "left")

# Top 20 by EB probability
top20 = (
    agg_named
    .orderBy(F.col("eb_p").desc())
    .select("merchant_abn", "merchant_name", "n_txn", "sum_amount", "eb_p", "EFL", "EFLR")
    .limit(20)
)
print("Top 20 merchants (highest eb_p):")
top20.show(20, truncate=False)

# Bottom 20 by EB probability
bottom20 = (
    agg_named
    .orderBy(F.col("eb_p").asc())
    .select("merchant_abn", "merchant_name", "n_txn", "sum_amount", "eb_p", "EFL", "EFLR")
    .limit(20)
)
print("Bottom 20 merchants (lowest eb_p):")
bottom20.show(20, truncate=False)

print("report merchants:", agg_named.count())
# ...existing code...

Top 20 merchants (highest eb_p):


+------------+-------------------------------+-----+------------------+-------------------+------------------+-------------------+
|merchant_abn|merchant_name                  |n_txn|sum_amount        |eb_p               |EFL               |EFLR               |
+------------+-------------------------------+-----+------------------+-------------------+------------------+-------------------+
|19492220327 |Commodo Ipsum Industries       |824  |8165222.509157542 |0.19062574634249185|2095977.484414645 |0.256695697155092  |
|90918180829 |Pharetra Quisque Company       |557  |5514144.783394542 |0.19022919073827627|1502177.7355237065|0.2724226139377785 |
|15043504837 |Odio Incorporated              |195  |3097642.3872911264|0.18459481843831577|905036.1621359454 |0.29216934977681375|
|14530561097 |Duis At Inc.                   |181  |2010872.9620992953|0.17681900463482017|570802.2527492613 |0.283857938073502  |
|44345785419 |Phasellus Nulla LLC            |89   |1346838.7511073337|0.1720239205

+------------+------------------------------+-----+------------------+-------------------+------------------+-------------------+
|merchant_abn|merchant_name                 |n_txn|sum_amount        |eb_p               |EFL               |EFLR               |
+------------+------------------------------+-----+------------------+-------------------+------------------+-------------------+
|82044452251 |Malesuada Integer Id Company  |1064 |1347129.1570628267|0.14711989670654177|191144.3904243432 |0.14189017394672032|
|43801715289 |Mollis Duis Sit Foundation    |414  |560035.8975183531 |0.1480073768921221 |81692.13521178266 |0.14586946225015068|
|85556646149 |Proin Nisl Inc.               |1956 |584194.1682930886 |0.14823168290392982|85489.85937114604 |0.14633809101677991|
|61085944703 |Vestibulum Mauris Institute   |397  |991123.518048757  |0.14835063321552577|141209.86756916545|0.1424745402542439 |
|82368304209 |Nec Incorporated              |5192 |9740349.300126279 |0.14839618676609265|

report merchants: 3925
Top 20 merchants (highest eb_p):


+------------+-------------------------------+-----+------------------+-------------------+------------------+-------------------+
|merchant_abn|merchant_name                  |n_txn|sum_amount        |eb_p               |EFL               |EFLR               |
+------------+-------------------------------+-----+------------------+-------------------+------------------+-------------------+
|19492220327 |Commodo Ipsum Industries       |824  |8165222.509157542 |0.19062574634249185|2095977.484414645 |0.256695697155092  |
|90918180829 |Pharetra Quisque Company       |557  |5514144.783394542 |0.19022919073827627|1502177.7355237065|0.2724226139377785 |
|15043504837 |Odio Incorporated              |195  |3097642.3872911264|0.18459481843831577|905036.1621359454 |0.29216934977681375|
|14530561097 |Duis At Inc.                   |181  |2010872.9620992953|0.17681900463482017|570802.2527492613 |0.283857938073502  |
|44345785419 |Phasellus Nulla LLC            |89   |1346838.7511073337|0.1720239205

+------------+------------------------------+-----+------------------+-------------------+------------------+-------------------+
|merchant_abn|merchant_name                 |n_txn|sum_amount        |eb_p               |EFL               |EFLR               |
+------------+------------------------------+-----+------------------+-------------------+------------------+-------------------+
|82044452251 |Malesuada Integer Id Company  |1064 |1347129.1570628267|0.14711989670654177|191144.3904243432 |0.14189017394672032|
|43801715289 |Mollis Duis Sit Foundation    |414  |560035.8975183531 |0.1480073768921221 |81692.13521178266 |0.14586946225015068|
|85556646149 |Proin Nisl Inc.               |1956 |584194.1682930886 |0.14823168290392982|85489.85937114604 |0.14633809101677991|
|61085944703 |Vestibulum Mauris Institute   |397  |991123.518048757  |0.14835063321552577|141209.86756916545|0.1424745402542439 |
|82368304209 |Nec Incorporated              |5192 |9740349.300126279 |0.14839618676609265|

report merchants: 3925
